Parameter optimization
================

This notebook shows how both hyperparameters and data source weights can be optimized. This code merely serves as a demonstration of the process. For the full optimization code see [run optimization](https://github.com/saeyslab/nichenetpy/blob/main/optimization/run_optimization.py) and [combine optimization folds](https://github.com/saeyslab/nichenetpy/blob/main/optimization/combine_optimization_folds.ipynb). 

In [1]:
from nichenetpy.utils import read_csv_cols
from nichenetpy.parameter_optimization import construct_and_evaluate
from nichenetpy.evaluation import EvaluationData

from optuna import create_study
from optuna.trial import Trial
from optuna.samplers import GPSampler
from itertools import chain

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

C:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in ("gr_human.csv", "lr_network_human.csv", "lr_sig_human.csv"):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
train_path = os.path.normpath("./tutorial_files/model_optimization")
if not os.path.exists(train_path):
    os.makedirs(train_path)
for filename in (
    "settings_training_f1234.json",
    "settings_training_f1235.json",
    "settings_training_f1245.json",
    "settings_training_f1345.json",
    "settings_training_f2345.json",
):
    file_path = os.path.join(train_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/15799578/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]
evaluation_data = EvaluationData(settings)

In [5]:
gr_network = gr_network[
    ~(
        (gr_network["database"] == "NicheNet_LT")
        & np.array([fr in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]])
    )
    & ~(
        (gr_network["database"] == "CytoSig")
        & np.array([fr in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]])
    )
]

In [6]:
source_names = sorted(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])))


def objective(trial: Trial):
    source_weights = {source_name: trial.suggest_float(name=source_name, low=0, high=1) for source_name in source_names}
    lr_sig_hub = trial.suggest_float(name="lr_sig_hub", low=0, high=1)
    gr_hub = trial.suggest_float(name="gr_hub", low=0, high=1)
    ltf_cutoff = trial.suggest_float(name="ltf_cutoff", low=0.9, high=0.999)
    damping_factor = trial.suggest_float(name="damping_factor", low=0.01, high=0.99)
    _, scores = construct_and_evaluate(
        source_weights,
        lr_sig_hub,
        gr_hub,
        ltf_cutoff,
        damping_factor,
        lr_network,
        gr_network,
        sig_network,
        evaluation_data,
    )
    return (scores["target_prediction"]["auroc"], scores["target_prediction"]["aupr_corrected"])


study = create_study(sampler=GPSampler(), directions=["maximize", "maximize"])
study.optimize(objective, n_trials=1, n_jobs=1)

[I 2026-11-26 10:56:15,046] A new study created in memory with name: no-name-d76ae428-4957-4189-a8ee-20cee4fb52d8


[I 2026-11-26 10:59:22,504] Trial 0 finished with values: [0.6325458885103493, 0.028721414843675096] and parameters: {'CytoSig_all': 0.7049481905613451, 'CytoSig_signature': 0.3186133924944414, 'HTRIDB': 0.07522514899788935, 'HuRi': 0.5867235827608523, 'HuRi_lit': 0.20973581974842959, 'HuRi_union_specific': 0.3252049239121556, 'KnockTF': 0.741671210065301, 'NicheNet_LT_frequent': 0.937954021959096, 'NicheNet_LT_infrequent': 0.7630148772493309, 'Remap_1': 0.9714161497449153, 'Remap_5': 0.373802989133875, 'cpdb_complex': 0.21938096443635513, 'cpdb_interaction': 0.3539124536548687, 'evex_binding': 0.22048740255258548, 'evex_catalysis': 0.9231439731201139, 'evex_phosphorylation': 0.3078511190487324, 'evex_regulation_binding': 0.905769409201006, 'evex_regulation_expression': 0.8002533224552102, 'evex_regulation_other': 0.8726049950274495, 'harmonizome_CHEA': 0.4468039074893544, 'harmonizome_DEPOD': 0.8841694481481077, 'harmonizome_ENCODE': 0.9935307307363299, 'harmonizome_GEO_GENE': 0.72223

the optimization results in a set of pareto optimal solutions

In [7]:
best_params = [trial.params for trial in study.best_trials]
len(best_params)

1

In [8]:
session_info.show()

C:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
